<a href="https://colab.research.google.com/github/somendrew/Fine-tuning_LLMs/blob/main/Fine_Tuning_Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.3 MB/s eta 0:00:00


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline, logging
from trl import SFTTrainer
from peft import LoraConfig, PeftModel, TaskType
import torch
import json
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)

In [6]:
with open("/content/science_10topics_100each.jsonl", "r") as f:
  data = [json.loads(line) for line in f if line.strip()]

# Convert to decoder-only format: input + output combined as one sequence
for d in data:
    output = d["output"]
    d["text"] = f"{d['input']}\n Question: {output['Question']}\n Answer: {output['Answer']}"

dataset = Dataset.from_list(data)

In [7]:
dataset

Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 1000
})

In [8]:
dataset[1]

{'input': 'Generate a one word question on Photosynthesis',
 'output': {'Answer': 'Sunlight',
  'Question': 'What energy source drives photosynthesis?'},
 'text': 'Generate a one word question on Photosynthesis\n Question: What energy source drives photosynthesis?\n Answer: Sunlight'}

## Load Model and Tokenizer

In [14]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

#tokenizer
base_model = AutoModelForCausalLM.from_pretrained(

  MODEL_NAME,
  device_map="auto",
  torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
  MODEL_NAME,
  trust_remote_code = True
  )

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_style = "right"


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [15]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

## BaseLine Generation

In [18]:
pipe = pipeline(
    'text-generation',
    model = base_model,
    tokenizer = tokenizer,
    max_length = 80
)

In [19]:
#prompt
ques_type = "multiple choice Question and Answer"
topic = "Physics"

prompt = f"Generate a {ques_type} on the topic {topic}"

result = pipe(prompt)
print(result[0]['generated_text'])

Generate a multiple choice Question and Answer on the topic Physics of the Universe. The question should require the students to identify the correct answer based on the given material.

1. Which force is responsible for attracting two objects together?

A) Newton's law of gravity
B) Electromagnetism
C) Quantum mechanics
D) Strong nuclear force


## LoRA Config